# 📦 LDA & QDA
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Linear and Quadratic Discriminant Analysis use probability theory to classify. Instead of fitting a boundary directly (like logistic regression), they model the distribution of X within each class and use Bayes' theorem to classify.

### What you'll learn
- Bayes' theorem as a classification framework
- LDA — linear decision boundaries (assumes shared covariance)
- QDA — quadratic boundaries (each class has its own covariance)
- When LDA/QDA outperform logistic regression
- Comparing all three: LR vs LDA vs QDA

### Dataset: Stock market direction prediction (Smarket)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Smarket

In [ ]:
import pandas as pd
smarket = pd.read_csv(f'{DATA_DIR}/Smarket.csv')
smarket['Direction_num'] = (smarket['Direction'] == 'Up').astype(int)
print(f"Smarket: {smarket.shape}  |  Up days: {smarket['Direction_num'].mean():.1%}")
smarket.head()

## 📐 Part 1 — Bayes' Theorem for Classification

LDA and QDA both use Bayes' theorem:

```
P(Y=k | X=x) = P(X=x | Y=k) × P(Y=k) / P(X=x)
             =   likelihood  ×   prior   / evidence
```

The class with the highest posterior probability wins.

**LDA** assumes Xₖ ~ N(μₖ, Σ) — each class has its own mean but **shared covariance** Σ. This gives *linear* decision boundaries.

**QDA** assumes Xₖ ~ N(μₖ, Σₖ) — each class has its own covariance Σₖ. This gives *quadratic* boundaries. More flexible but needs more data to estimate.

In [ ]:
# Visualize LDA vs QDA decision boundaries
from sklearn.datasets import make_classification
np.random.seed(42)

# Create two scenarios
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scenario 1: Equal covariance → LDA is optimal
X1 = np.vstack([np.random.multivariate_normal([0,0], [[1,0.5],[0.5,1]], 150),
                np.random.multivariate_normal([2,2], [[1,0.5],[0.5,1]], 150)])
y1 = np.array([0]*150+[1]*150)

# Scenario 2: Unequal covariance → QDA wins
X2 = np.vstack([np.random.multivariate_normal([0,0], [[0.5,0],[0,2]], 150),
                np.random.multivariate_normal([2,2], [[2,0],[0,0.5]], 150)])
y2 = np.array([0]*150+[1]*150)

for ax, X, y, title in zip(axes, [X1,X2], [y1,y2],
                            ['Equal covariances → LDA wins', 'Unequal covariances → QDA wins']):
    lda = LinearDiscriminantAnalysis().fit(X, y)
    qda = QuadraticDiscriminantAnalysis().fit(X, y)
    
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200),
                          np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    Z_lda = lda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_lda, colors=['#1e5fa8'], linewidths=2, linestyles='-')
    
    Z_qda = qda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_qda, colors=['#e85d20'], linewidths=2, linestyles='--')
    
    ax.scatter(X[y==0,0], X[y==0,1], color='#1e5fa8', alpha=0.4, s=15)
    ax.scatter(X[y==1,0], X[y==1,1], color='#e85d20', alpha=0.4, s=15)
    
    from matplotlib.lines import Line2D
    ax.legend([Line2D([0],[0],color='#1e5fa8',lw=2),
               Line2D([0],[0],color='#e85d20',lw=2,ls='--')],
              [f'LDA (acc={lda.score(X,y):.2f})',
               f'QDA (acc={qda.score(X,y):.2f})'], fontsize=9)
    ax.set_title(title)

plt.suptitle('LDA vs QDA Decision Boundaries', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 LDA boundary is always a straight line. QDA can curve to fit unequal covariances.")

In [ ]:
# Compare LR, LDA, QDA on Smarket
predictors = ['Lag1','Lag2','Lag3','Lag4','Lag5','Volume']
X = smarket[predictors].values
y = smarket['Direction_num'].values

# Train on 2001-2004, test on 2005 (temporal split — important for market data!)
mask_train = smarket['Year'] < 2005
X_tr, X_te = X[mask_train], X[~mask_train]
y_tr, y_te = y[mask_train], y[~mask_train]

print(f"Train: {X_tr.shape[0]} days (2001-2004)")
print(f"Test: {X_te.shape[0]} days (2005)")

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'LDA':                 LinearDiscriminantAnalysis(),
    'QDA':                 QuadraticDiscriminantAnalysis(),
}

print("\n=== Accuracy on 2005 Test Data ===")
for name, model in models.items():
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    print(f"  {name:<25} {acc:.4f}")

print("\n📌 All three methods are comparable here — the stock market is hard to predict!")
print("   Baseline (always predict 'Up'): {:.4f}".format(y_te.mean()))

In [ ]:
# LDA's discriminant scores — visualize the separation
lda = LinearDiscriminantAnalysis()
lda.fit(X_tr, y_tr)

scores_tr = lda.transform(X_tr)
scores_te = lda.transform(X_te)

fig, ax = plt.subplots(figsize=(9, 4))
for split, scores, y_split, label, alpha in [
    (None, scores_tr, y_tr, 'Train', 0.4),
    (None, scores_te, y_te, 'Test', 0.9)]:
    ax.hist(scores[y_split==0, 0], bins=25, alpha=alpha*0.7,
            color='#1e5fa8', density=True,
            label=f'Down ({label})' if alpha > 0.5 else None)
    ax.hist(scores[y_split==1, 0], bins=25, alpha=alpha*0.7,
            color='#e85d20', density=True,
            label=f'Up ({label})' if alpha > 0.5 else None)

ax.axvline(0, color='black', lw=1.5, ls='--', label='Decision boundary')
ax.set_xlabel('LDA Discriminant Score')
ax.set_ylabel('Density')
ax.set_title('LDA Discriminant Scores — Up vs Down Days\n(overlap = hard classification problem)')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Large overlap between Up/Down scores confirms stock market is hard to predict from lagged returns")

In [ ]:
# @title 📝 Quiz — LDA & QDA
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What distributional assumption does LDA make about the features?
# @markdown **Q2:** What is the key difference between LDA and QDA?
# @markdown **Q3:** When does LDA outperform logistic regression?
# @markdown **Q4:** Why do we need a prior P(Y=k) in Bayes' theorem?
# @markdown **Q5:** If your data has 5 classes and 2 predictors, how many discriminant functions does LDA produce?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_="LDA & QDA"
# @title 🤖 AI Grading
# @markdown **Enter your GitHub username, then click ▶ Run.**
# @markdown Colab will send your answers to Gemini automatically — no key needed.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

import textwrap
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {v or '(no answer)'}"
                    for i,(_,v) in enumerate(answers.items(),1))

    prompt = (f"Grade these quiz answers for the \"{_NB_TITLE}\" notebook "
              f"(Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
              f"{qa}\n\n"
              f"For each answer say CORRECT, PARTIAL, or INCORRECT and one sentence why. "
              f"Accept any reasonable paraphrase as correct. "
              f"End with an overall grade (Excellent/Good/Needs Review/Incomplete) "
              f"and one sentence on what to review if they struggled.")

    # Try Colab's built-in Gemini via the generativeai SDK (no key needed in Colab)
    success = False
    try:
        import google.generativeai as genai
        # In Colab, calling generate_content without configure() uses
        # Colab's own managed credentials automatically
        model = genai.GenerativeModel("gemini-1.5-flash")
        resp  = model.generate_content(prompt)
        print("\n" + "\u2550"*56)
        print(f"  \U0001f916 Feedback \u2014 {_NB_TITLE}")
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}")
        print("\u2550"*56)
        for line in resp.text.strip().split("\n"):
            if line.strip():
                for w in textwrap.wrap(line.strip(), 54):
                    print(f"  {w}")
            else:
                print()
        print("\u2550"*56)
        success = True
    except Exception:
        pass

    if not success:
        # Fallback: print the prompt so they can paste it into Colab's AI chat
        print("\u2550"*56)
        print("  Automatic grading unavailable.")
        print("  Paste the text below into the Gemini chat panel:")
        print("  (click the \u2728 sparkle icon in the Colab toolbar)")
        print("\u2550"*56)
        print()
        print(prompt)
        print()
        print("\u2550"*56)

    print("\n  \U0001f4be File \u2192 Save a copy in GitHub \u2192 your fork")
